## Importando as bibliotecas

In [232]:
!pip install surprise


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [233]:
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader, KNNBasic, SVDpp, KNNBaseline, NMF
from surprise.model_selection import train_test_split
from surprise import accuracy
import re
import os

## Importando os dados

In [234]:
ratings = pd.read_csv("data/ratings.csv")

In [235]:
anime = pd.read_csv("data/anime_data.csv")

In [236]:
ratings = ratings[ratings['user_id'] <= 6000]

## Primeiro Modelo: SVD

In [237]:
reader = Reader(rating_scale=(1,10))

In [238]:
data_svd = Dataset.load_from_df(ratings, reader)

In [239]:
trainset, testset = train_test_split(data_svd, test_size=0.2, random_state=42)

In [240]:
model_svd = SVD()
model_svd.fit(trainset)

In [241]:
predictions = model_svd.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 1.2481


1.248088333330507

In [242]:
mae = accuracy.mae(predictions)
mae

MAE:  0.9229


0.922916240374237

## 2° Modelo: KNNBasic

In [243]:
sim_options = {
    "name": "cosine",
    "user_based": True  
}

model_knn = KNNBasic(sim_options=sim_options)
model_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [244]:
predictions = model_knn.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 1.5511


1.5510965097223477

In [245]:
mae = accuracy.mae(predictions)
mae

MAE:  1.1639


1.163860473141019

## 3° Modelo: SVD++

In [246]:
model_svdpp = SVDpp()

model_svdpp.fit(trainset)

In [247]:
predictions = model_svdpp.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 1.2422


1.2422250033359394

In [248]:
mae = accuracy.mae(predictions)
mae

MAE:  0.9158


0.9157803063506563

## 4° Modelo: KNN Baseline

In [249]:
model_baseline = KNNBaseline()

model_baseline.fit(trainset)

Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.


In [250]:
predictions = model_baseline.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 1.2571


1.257066215934165

In [251]:
mae = accuracy.mae(predictions)
mae

MAE:  0.9340


0.9340087428552164

## 5° Modelo: NMF

In [252]:
model_nmf = NMF()

model_nmf.fit(trainset)

In [253]:
predictions = model_nmf.test(testset)
rmse = accuracy.rmse(predictions)
rmse

RMSE: 2.0686


2.0685938965935766

In [254]:
mae = accuracy.mae(predictions)
mae

MAE:  1.7967


1.7966832577974492

## Recomendando para usuários

In [255]:
not_first_keywords = [
    "2nd", "3rd", "4th", "5th", "6th", "7th"
    "II ", "III", "IV", "V", "VI", "VII"
    "Season 1", "Season 2", "Season 3", "Season 4", "Season 5", "Season 6", "Season 7"
    "Second Season", "Third Season", "Fourth Season", "Final Season",
    "Part 2", "Part 3", "Part 4", "Part 5", "Part 6", "Part 7",
    "2nd Season", "3rd Season", "4th Season", "5th Season", "6th Season", "7th Season",
    "Special", "The Final", "Movie",
    "2", "3", "4",
    "Second Season", "Third Season", "Fourth Season", "Fifth Season"
]

def is_first_season(title):
    title_lower = title.lower()
    return not any(k.lower() in title_lower for k in not_first_keywords)

anime = anime[anime['title'].apply(is_first_season)]

In [256]:
cleaned = [
    title for title in anime['title']
    if not (
        (title.startswith("Gintama") and title != "Gintama") or
        (title.startswith("One Piece") and title != "One Piece") or
        (title.startswith("Natsume Yuujinchou") and title != "Natsume Yuujinchou") or
        (title.startswith("Cowboy Bebop") and title != "Cowboy Bebop") or
        (title.startswith("Tian Guan Cifu") and title != "Tian Guan Cifu") or
        (title.startswith("Mo Dao Zu Shi") and title != "Mo Dao Zu Shi") or
        (title.startswith("Kaguya-sama wa Kokurasetai") and title != "Kaguya-sama wa Kokurasetai") or 
        (title.startswith("Shouwa Genroku Rakugo Shinjuu") and title != "Kaguya-sama wa Kokurasetai") or
        (title.startswith("Baki") and title != "Baki") or 
        (title.startswith("Initial D") and title != "Initial D") or
        (title.startswith("Bleach") and title != "Bleach") or 
        (title.startswith("Naruto") and title != "Naruto") or 
        (title.startswith("Mob Psycho 100") and title != "Mob Psycho 100") or
        (title.startswith("Shiguang Dailiren") and title != "Shiguang Dailiren") or 
        (title.startswith("Mushishi") and title != "Mushishi") or 
        (title.startswith("Code Geass: Hangyaku no Lelouch") and title != "Code Geass: Hangyaku no Lelouch") or
        (title.startswith("Clannad") and title != "Clannad")
    )
]

anime = anime[anime['title'].isin(cleaned)]


In [257]:
def recommend_to_user(model, anime, user_data, user_id=-1, n=10):
    watched = set(user_data[user_data["user_id"] == user_id]["anime_id"])

    all_animes = set(anime["mal_id"].unique())

    candidates = all_animes - watched

    predictions = []
    for anime_id in candidates:
        est = model.predict(user_id, anime_id).est
        predictions.append((anime_id, est))

    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]


In [258]:
def reccomendation_file(recommendations, anime):
    result = []
    for anime_id, score in recommendations:
        title = anime.loc[anime["mal_id"] == anime_id, "title"].values
        title = title[0] if len(title) else "Unknown"
        result.append((title))
    return result


### Gerando as recomendações

In [259]:
folder = "data/test_lists"

In [260]:
models = [
    ("svd", model_svd),
    ("knn", model_knn),
    ("svd++", model_svdpp),
    ("KNNBaseline", model_baseline),
    ("nmf", model_nmf),
]

In [263]:
for filename in os.listdir(folder):
    if not filename.endswith(".xml"):
        continue

    xml_file = os.path.join(folder, filename)

    user_test = pd.read_xml(xml_file)
    user_test = user_test[['my_id','series_animedb_id', 'my_score']]
    user_test = user_test.rename(columns={
        'my_id': 'user_id',
        'series_animedb_id': 'anime_id',
        'my_score': 'rating'
    })
    user_test['user_id'] = -1  

    user_name = filename.replace(".xml", "")

    for model_name, model in models:

        recs = recommend_to_user(model, anime, user_test)

        recs = reccomendation_file(recs, anime)

        output_name = f"data/reccomendations/{user_name}_{model_name}.txt"

        with open(output_name, "w", encoding="utf-8") as f:
            for item in recs:
                f.write(str(item) + "\n")

        print(f"Saved: {output_name}")


Saved: data/reccomendations/Sylvio_svd.txt
Saved: data/reccomendations/Sylvio_knn.txt
Saved: data/reccomendations/Sylvio_svd++.txt
Saved: data/reccomendations/Sylvio_KNNBaseline.txt
Saved: data/reccomendations/Sylvio_nmf.txt
Saved: data/reccomendations/Rain_svd.txt
Saved: data/reccomendations/Rain_knn.txt
Saved: data/reccomendations/Rain_svd++.txt
Saved: data/reccomendations/Rain_KNNBaseline.txt
Saved: data/reccomendations/Rain_nmf.txt
Saved: data/reccomendations/LuizFelix_svd.txt
Saved: data/reccomendations/LuizFelix_knn.txt
Saved: data/reccomendations/LuizFelix_svd++.txt
Saved: data/reccomendations/LuizFelix_KNNBaseline.txt
Saved: data/reccomendations/LuizFelix_nmf.txt
Saved: data/reccomendations/Bluuq_svd.txt
Saved: data/reccomendations/Bluuq_knn.txt
Saved: data/reccomendations/Bluuq_svd++.txt
Saved: data/reccomendations/Bluuq_KNNBaseline.txt
Saved: data/reccomendations/Bluuq_nmf.txt
Saved: data/reccomendations/NekoCat_svd.txt
Saved: data/reccomendations/NekoCat_knn.txt
Saved: data/